# Part 3. Retrieval-Augmented Generation (RAG)

- **Retrieval-Augmented Generation (RAG)** combines **information retrieval** with **language generation**, enabling LLMs to produce more accurate and grounded responses by incorporating external knowledge sources.

- Instead of relying solely on pre-trained knowledge, RAG systems **retrieve relevant documents** from a database and **inject them into the prompt** before generating a response.

#### Components of a RAG Pipeline
- **Document Store**: A collection of texts or data to retrieve from.
- **Embedding Model**: Converts text into vector representations for similarity search.
- **Retriever**: Finds the most relevant documents based on a query.
- **Augmented Prompt**: Combines retrieved context with the user query.
- **LLM Generator**: Produces the final answer using the enriched prompt.

In [1]:
# # Start our ollama server in the background to host our LLMs
from ollama_utils import start_ollama_server, stop_ollama_server, generate_with_ollama

# start ollama server
start_ollama_server()

🚀 Starting Ollama server...
📄 Server logs: ollama_server.log
📍 API endpoint: http://localhost:11434
⏳ Waiting 5 seconds for server startup...
✅ Ollama server ready! PID: 2271756


## Building the RAG Knowledge Base

- In this step, we construct a **vector database** that allows our LLM to retrieve relevant information at query time.

- We begin by defining a small corpus of text (e.g., research abstracts and schedule data), which will serve as the **external knowledge source** for our RAG system.

#### Key Steps in This Process
- **Document Creation**: Convert raw text into structured `Document` objects for indexing.
- **Embedding Generation**: Use a Hugging Face embedding model to transform text into vector representations.
- **Vector Indexing**: Store embeddings in a `VectorStoreIndex` for efficient similarity search.
- **Retriever Setup**: Configure a retriever to return the top-\(k\) most relevant documents (here, \(k = 3\)) for a given query.

In [2]:
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ------------------ Build RAG Database ------------------
# Let's build a database of relevant information we want our llm to draw from

# abstract text from an astronomy paper (Jeong-Eun Lee et al. 2023 ApJ 953 82)
abstract1 = "Most stars form in multiple-star systems. For a better understanding of their formation processes, it is important to resolve the individual protostellar components and the surrounding envelope and disk material at the earliest possible formation epoch, because the formation history can be lost in a few orbital timescales. Here we present Atacama Large Millimeter/submillimeter Array observational results of a young multiple protostellar system, IRAS 04239+2436, where three well-developed large spiral arms were detected in the shocked SO emission. Along the most conspicuous arm, the accretion streamer was also detected in the SO2 emission. The observational results are complemented by numerical magnetohydrodynamic simulations, where those large arms only appear in magnetically weakened clouds. Numerical simulations also suggest that the large triple spiral arms are the result of gravitational interactions between compact triple protostars and the turbulent infalling envelope."

# abstract text from a laser sail paper (Gabriel R. Jaffe et al. Nano Lett. 2023, 23, 15, 6852–6858)
abstract2 = "Laser sails propelled by gigawatt-scale ground-based laser arrays have the potential to reach relativistic speeds, traversing the solar system in hours and reaching nearby stars in years. Here, we describe the danger interplanetary dust poses to the survival of a laser sail during its acceleration phase. We show through multiphysics simulations how localized heating from a single optically absorbing dust particle on the sail can initiate a thermal runaway process that rapidly spreads and destroys the entire sail. We explore potential mitigation strategies, including increasing the in-plane thermal conductivity of the sail to reduce the peak temperature at hot spots and isolating the absorptive regions of the sail that can burn away individually."

# the schedule for today's tutorial
day4schedule = "Morning Session: 9am - 11:45am, Lunch: 11:45am - 1pm, Afternoon Sessions will run from 1pm - 4pm"

# Prepare our text data into llama_index Document objects
abstract_list = [abstract1, abstract2, day4schedule]
documents = [Document(text=text) for text in abstract_list]

# Create an indexed vector database of our documents
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# Create retriever for top 1 document
retriever = index.as_retriever(similarity_top_k=3)
# ----------------------------------------------------------

## Running the RAG Pipeline End-to-End

This cell executes the full **Retrieval → Augmentation → Generation loop** over multiple queries and logs the results for inspection.

We have three prompts asking about different topics which the base foundation model does not know the answer too.  For each prompt, we'll first retrieve the most relevant document from our vectorstore, then add the document text to our original prompt and then have the LLM answer. For now we're simply selecting the document with the highest similarity score.

For more advanced retrieval techniques, you could select documents based on a similarity score threshold, or even have send the retrieved documents to an LLM to score for relevancy to the topic at hand.  For now, let's see if our LLM augmented with a RAG pipeline can correctly answer questions about the two papers and the lunch schedule we included in our vectorstore.

In [4]:
from pprint import pprint
import json
import pandas as pd

prompts = [
    "How many arms does IRAS 04239+2436 have?",
    "How do I protect my laser sail against zodiacal dust? Respond with 1-2 sentences.",
    "when's lunch?",
]

results = []

for prompt in prompts:
    print("\n" + "=" * 100)
    print("PROMPT:")
    print(prompt)

    # Retrieve top k=3 entries
    retrieved_nodes = retriever.retrieve(prompt)

    # Extract text + scores
    retrieved_entries = []
    for i, node in enumerate(retrieved_nodes[:3], 1):
        text = getattr(node, "text", str(node))
        score = getattr(node, "score", None)

        if score is None:
            score = getattr(node, "similarity", None)

        if score is None and hasattr(node, "metadata") and node.metadata:
            score = node.metadata.get("score") or node.metadata.get("similarity")

        retrieved_entries.append({
            "rank": i,
            "text": text,
            "score": score,
        })

    top_entry = retrieved_entries[0]["text"] if retrieved_entries else ""

    print("\nTOP RETRIEVED VECTOR DB ENTRY:")
    print(top_entry)

    print("\nRETRIEVED DATABASE ENTRIES WITH SIMILARITY SCORES:")
    for entry in retrieved_entries:
        print(f"[{entry['rank']}] score={entry['score']} | {entry['text']}")

    # Inject only top 1 into the prompt
    augmented_prompt = f"*Background Context*\n{top_entry}\n\n*Query*\n{prompt}"

    print("\nFULL LLM PROMPT:")
    print(augmented_prompt)

    response = generate_with_ollama(
        model="gemma3:1b",
        user_prompt=augmented_prompt,
        verbose=False,
    )

    print("\nLLM RESPONSE:")
    print(response.get("response", response))

    results.append({
        "prompt": prompt,
        "top_retrieved_vector_db_entry": top_entry,
        "retrieved_database_entries": json.dumps(retrieved_entries, ensure_ascii=False),
        "full_llm_prompt": augmented_prompt,
        "llm_response": response.get("response", response),
        "model": response.get("model"),
        "context_length": response.get("context_length"),
        "total_duration": response.get("total_duration"),
        "wall_time": response.get("wall_time"),
        "prompt_eval_count": response.get("prompt_eval_count"),
        "eval_count": response.get("eval_count"),
        "tokens_per_second": response.get("tokens_per_second"),
    })

df = pd.DataFrame(results)


PROMPT:
How many arms does IRAS 04239+2436 have?

TOP RETRIEVED VECTOR DB ENTRY:
Most stars form in multiple-star systems. For a better understanding of their formation processes, it is important to resolve the individual protostellar components and the surrounding envelope and disk material at the earliest possible formation epoch, because the formation history can be lost in a few orbital timescales. Here we present Atacama Large Millimeter/submillimeter Array observational results of a young multiple protostellar system, IRAS 04239+2436, where three well-developed large spiral arms were detected in the shocked SO emission. Along the most conspicuous arm, the accretion streamer was also detected in the SO2 emission. The observational results are complemented by numerical magnetohydrodynamic simulations, where those large arms only appear in magnetically weakened clouds. Numerical simulations also suggest that the large triple spiral arms are the result of gravitational interactions 